# Phase 5 (LSI) - Model 3 (MTP) + LoRMA fine-tuning, every chunk of every document, hierarchical aggregation

This is the Model-3 all-chunks notebook with one change: the backbone is fine-tuned with **LoRMA** (Low-Rank *Multiplicative* Adaptation) instead of LoRA. Everything else - the 1024-token windows, the hierarchical document head, the loss, EMA, thresholds, resume logic and the submission format - is identical to the Model-3 LoRA notebook, so the two dev tables are directly comparable.

**Kaggle setup (do this before running)**
1. *Add Data*: (a) the Phase-1 **Model-3** pretraining output (`best.pt` + `best_meta.json`, tag `decoder_only_rope_gqa_swiglu_mtp4_v1`; checkpoints of the other Phase-1 models carry a different tag and are ignored), (b) the tokenizer output you pretrained Model 3 with (`vocab_50k.json` + `merges_50k.json`, and/or the byte-level `tokenizer.json` from the BPE notebook). Everything is found automatically under `/kaggle/input`.
2. *Settings*: Accelerator = GPU (T4), Internet = ON. Add a Kaggle secret named `HF_TOKEN` (a Hugging Face token with access to the gated dataset; `HF`, `HUGGINGFACE_TOKEN` also work) and tick it for this notebook. The token is read from the secret, never from the code - do not paste a token into a cell.
3. Run once with `SMOKE_TEST = True` in cell 1 (tiny subset, 1 epoch) to verify the pipeline and to read the real training speed, then set it to `False` and run everything.
4. Output: `/kaggle/working/p5_lsi_m3_lorma_v1_allchunks/submission/submission.zip` (+ `prediction.json`, `dev_probs.npy`, `test_probs.npy`, `thresholds_and_report.json`; the report has a `fine_tuning` section with the exact adapter settings).
5. If the session dies / the time budget ends: *Save & Commit*, attach that notebook's output as an input of the next run, run again - training resumes automatically (weights, optimizer, EMA, batch position and the used-up training time are restored). A checkpoint written with other fine-tuning settings (method, variant, rank, alpha, targets) is refused with a clear message instead of being loaded wrongly. To train longer after a finished run, raise `TRAIN_HOURS`.

**LoRMA (cell 1 + 2)**
- **What it is.** LoRA *adds* a low-rank term to a frozen weight: `W = W0 + s·B·A`. LoRMA (Bihany, Patel & Modi, Findings of ACL 2025) *multiplies* the frozen weight by a low-rank-adapted matrix instead: `W = M · W0`. A purely low-rank `M` would cap the rank of `M·W0` at `r`, so the paper adds a **rank-inflation** step (the identity in LoRMA+, a permutation in LoRMA-π) and reorders the computation so the large `d_out × d_out` matrix is not needed; the update `M·W0 − W0` lives in the row space of `W0`.
- **LoRMA+ (default, `LORMA_VARIANT = "plus"`).** `M = I + s·B·A` with `B: d_out × r`, `A: r × d_out`, `s = alpha / r`. `B = 0` at the start, so `M = I` and the backbone starts at exactly the pretrained function. It is computed as `y = W0·x + s·B(A(W0·x))`, so the `d_out × d_out` matrix `M` is never formed in the forward pass.
- **LoRMA-π (`LORMA_VARIANT = "pi"`).** `M = P(B·A)`, where row `i` of `B·A` is cyclically shifted by `i` (a permutation instead of the identity); no `alpha`, no adapter dropout; initialised so that `M = I`; the adapter weights are not weight-decayed (decay would shrink `M` towards 0). It needs a dense `d_out × d_out` multiplication in every adapted layer, so expect it to be **noticeably slower** than LoRMA+. The default is LoRMA+.
- **Targets and size.** `q/k/v/o` of the attention and `w_gate/w_up/w_down` of the SwiGLU feed-forward in every block; the pretrained weights stay frozen (the MTP-head-1 norm is trained too). Defaults `r = 32`, `alpha = 64` (scale 2.0), `LORA_LR = 4e-4` (the value of the LoRA notebook, not re-tuned for LoRMA); the log line "LoRMA+ modules: ..." prints the number of adapted layers and trainable parameters.
- **EMA and checkpoints.** The EMA (used for the dev score and the best checkpoint) is kept on the trainable tensors: the factors `A` and `B` and the head. It is not an average of the product `M·W0`, which is the same behaviour as in the LoRA notebook. Checkpoints store only the trainable tensors, so the Phase-1 weights must be attached again when resuming.
- **Switch.** `FT_METHOD = "lora"` (or `"rslora"` with `LORA_R = 64`, `LORA_ALPHA = 16`) gives the earlier LoRA behaviour in this same notebook; use a new `RUN_TAG` when you change it.
- **Not tuned.** `r`, `alpha`, `LORA_LR` and `TRAIN_HOURS` are sensible starting values, not tuned on your data. Multiplicative updates act on the whole output vector of a layer, so if the loss spikes in the first epoch, lower `LORA_LR` (e.g. `2e-4`) or `LORA_ALPHA`.

**What is specific to Model 3**
- The backbone (RMSNorm, RoPE, GQA, SwiGLU, MTP heads) is extracted *verbatim* from the Model-3 notebook; its architecture is read from the checkpoint's saved `config` (not from constants), so it follows whatever you actually trained.
- Model 3 has no `final_norm`. Its features come from MTP head 1, `RMSNorm(x + W x)`; that is the chunk representation here. The t+2 / t+3 heads are dropped after loading.
- **Tokenizer check**: the Model-3 notebook loads a custom word-level BPE (`vocab_50k.json`, lower-cased), while the BPE notebook writes a byte-level `tokenizer.json`. Cell 4 finds out which one the checkpoint was pretrained with by running the *pretrained model itself* on a few documents: the right tokenizer reproduces the pretraining `val_loss`, a wrong one gives ~7+. It prints both numbers, uses the better one, and stops with a clear message if neither fits.
- The custom tokenizer's `WORD_PATTERN` has a character-class range (`\a-z`) that drops ASCII punctuation. It is reproduced exactly because the checkpoint was pretrained on that token stream.

**Method (same as the Model-3 LoRA and Model-2 all-chunk notebooks)**
- Every document is cut into ALL overlapping 1024-token windows (stride 768 - the Model-3 pretraining geometry). Training, dev and test encode every window; no sampling, no character truncation (only a 40k-token safety cap, ~52 chunks).
- Chunk embedding `[mean-pool ; last-token]` -> MLP -> chunk-index and relative-position embeddings -> a small transformer over the chunk sequence (section-aware) -> three document-level aggregations, each using every chunk: (a) label-wise attention, (b) chunk-level statute logits with log-mean-exp pooling, (c) mean of chunk vectors. A learned per-statute gate fuses them; each path also has its own auxiliary BCE loss. The last cell prints an ablation of the three paths on dev.
- Loss: independent sigmoids, BCE with capped `pos_weight` for rare statutes. Decision: per-label / global / blended thresholds chosen by cross-fitted dev macro-F1, at least one statute per document.
- Free-tier friendly: time-aware cosine LR (`TRAIN_HOURS = 7.5` in total, `EPOCHS = 3` as the upper bound), EMA weights, resumable checkpoints, best epoch kept by dev macro-F1. The low-rank branch of LoRMA+ is only a few percent of the cost of an adapted layer, so it should run at about the speed of the LoRA notebook (not measured on a T4 - read the epoch ETA of the smoke test); LoRMA-π is slower.

**Things to know**
- The numbers that matter are the dev tables at the end (cross-fitted thresholds), not the training loss.
- This is one submission from one Phase-1 model (Model 3). The task needs your best four submissions from four *different* Phase-1 models; `dev_probs.npy` / `test_probs.npy` are saved so the models can also be ensembled.

**Source:** H. Bihany, S. Patel, A. Modi, *LoRMA: Low-Rank Multiplicative Adaptation for LLMs*, Findings of ACL 2025 (arXiv 2506.07621). This notebook's implementation is written from the paper's description, not taken from the authors' code.

In [ ]:
# ============================================================
# PHASE 5 (LSI) | MODEL 3 (RoPE+GQA+SwiGLU, 3 MTP heads, BPE-50k) + LoRMA (multiplicative low-rank adaptation)
# ALL-CHUNK hierarchical model: every overlapping chunk of every document is encoded (train / dev / test)
# CELL 1: CONFIG + HELPERS
# ============================================================
import os, re, sys, gc, json, math, time, random, shutil, zipfile, hashlib, contextlib
from dataclasses import dataclass, asdict, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# ------------------------------------------------------------------
# SWITCH: run once with True (tiny subset, 1 epoch, ~5-10 min) to verify the whole pipeline,
# then set False for the real run.
# ------------------------------------------------------------------
SMOKE_TEST = False

RUN_TAG = "p5_lsi_m3_lorma_v1_allchunks" + ("_smoke" if SMOKE_TEST else "")
PHASE1_MODEL_TAG = "decoder_only_rope_gqa_swiglu_mtp4_v1"      # MODEL_TAG written by the Model-3 notebook

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK_DIR = ("/kaggle/working" if IS_KAGGLE else ".") + f"/{RUN_TAG}"
CKPT_DIR = f"{WORK_DIR}/checkpoints"
CACHE_DIR = f"{WORK_DIR}/cache"
OUT_DIR = f"{WORK_DIR}/submission"
for _d in (CKPT_DIR, CACHE_DIR, OUT_DIR):
    os.makedirs(_d, exist_ok=True)

# Optional manual overrides. Leave "" to auto-search /kaggle/input.
TOKENIZER_DIR = ""      # restrict the tokenizer search to this folder
PHASE1_CKPT = ""        # full path of the Model-3 best.pt (Phase-1 pretraining output)

# ---- tokenizer selection ----
# The Model-3 notebook loads a custom word-level BPE (vocab_50k.json + merges_50k.json, lower-cased, '</w>' markers); the
# BPE notebook writes a byte-level tokenizer.json. Whichever one the checkpoint was pretrained with is found by asking the
# pretrained LM itself: the right tokenizer gives a next-token loss close to its pretraining val_loss, a wrong one ~7+.
TOKENIZER_KIND = "auto"           # "auto" | "custom" (word-BPE) | "hf" (byte-level tokenizer.json)
TOKENIZER_CHECK = True            # run the LM-loss check (needs the backbone on the GPU for a few seconds)
TOKENIZER_STRICT = True           # raise if even the best candidate is far worse than the pretraining val_loss
LM_CHECK_DOCS = 16                # documents used by the check (first CHUNK_LEN tokens of each)
LM_CHECK_MARGIN = 3.0             # allowed excess over the pretraining val_loss before STRICT raises

# ---- data (same HF repo / files as your EDA notebook) ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE5_FILES = {
    "train": "Phase_5_LSI/Train/train.json",
    "dev":   "Phase_5_LSI/Train/dev.json",
    "test":  "Phase_5_LSI/Test/test.json",
}

# ---- text -> tokens ----
SIMPLIFY_PLACEHOLDERS = True      # "<ENTITY>" -> "ENTITY" before tokenising (anonymised corpus)
MAX_DOC_TOKENS = 40_000           # pure safety cap (~52 chunks, far above the p99 doc): head 70% + tail 30% are kept.
KEEP_HEAD_FRAC = 0.7              # No character truncation: practically every token of every document is used.

# ---- overlapping chunking: EVERY window of the document is used (no random / evenly-spaced subset) ----
# (same 1024 / 768 geometry as the Model-3 pretraining windows)
CHUNK_LEN = 1024
STRIDE = 768                      # 256-token overlap between consecutive chunks
OVERLAP = CHUNK_LEN - STRIDE
ENC_MB_TRAIN = 32                 # max chunks pushed through the backbone at once in training (memory guard only:
ENC_MB_EVAL = 64                  # ALL chunks of a document are still in the graph / in the prediction)

# ---- fine-tuning strategy of the backbone  -  THE ONE THING THAT DIFFERS from the Model-3 LoRA notebook ----
FT_METHOD = "lorma"               # "lorma" (this notebook) | "lora" (= the earlier Model-3 LoRA notebook) | "rslora" (then use LORA_R = 64, LORA_ALPHA = 16)
LORA_R = 32                       # rank of the adapters
LORA_ALPHA = 64                   # scale: lora / lorma+ = alpha / r (r = 32, alpha = 64 -> 2.0) | rslora = alpha / sqrt(r)
USE_RSLORA = (FT_METHOD == "rslora")
# LoRMA = Low-Rank Multiplicative Adaptation (Bihany, Patel & Modi, Findings of ACL 2025): the adapter MULTIPLIES the frozen weight,
#   W = M W0  with  M = I + s B A  (B: d_out x r, A: r x d_out)   instead of adding to it (LoRA: W = W0 + s B A).
LORMA_VARIANT = "plus"            # "plus" = LoRMA+ (rank inflation by adding the identity; s = alpha / r) | "pi" = LoRMA-pi (rank inflation by a cyclic row
                                  # permutation instead of the identity; no alpha, no adapter dropout; slower: a dense d_out x d_out product per layer)
LORA_DROPOUT = 0.05
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
assert FT_METHOD in ("lorma", "rslora", "lora"), f"FT_METHOD must be lorma / rslora / lora, got {FT_METHOD!r}"
assert LORMA_VARIANT in ("plus", "pi"), f"LORMA_VARIANT must be plus / pi, got {LORMA_VARIANT!r}"
FT_DROPOUT = 0.1                  # residual dropout inside the backbone during fine-tuning (Model-3 pretraining: 0.1)

# ---- hierarchical document head ----
HEAD_HIDDEN = 512                 # chunk embedding size inside the head
HEAD_DROPOUT = 0.1
DOC_LAYERS = 1                    # transformer layers over the chunk sequence (chunks exchange information)
DOC_HEADS = 8
DOC_FF = 1024
MAX_POS_EMB = 64                  # learned chunk-index embeddings (needs n_chunks <= 64)
REL_BUCKETS = 8                   # "where in the document" buckets (+1 extra bucket for single-chunk docs)
AUX_W = 0.3                       # weight of the per-path auxiliary BCE losses (deep supervision of the 3 paths)
POS_WEIGHT_CAP = 5.0              # pos_weight = clip(sqrt(neg/pos), 1, cap)

# ---- optimisation ----
EPOCHS = 3
TRAIN_TOKEN_BUDGET = 24 * CHUNK_LEN   # max (chunks x chunk_len) per optimizer step; a longer doc goes alone in a batch
EVAL_TOKEN_BUDGET = 64 * CHUNK_LEN
MAX_DOCS_PER_BATCH = 48
DOC_NORM = 16.0                       # loss = sum over docs / DOC_NORM  (every doc weighs the same)
LORA_LR = 4e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
MIN_LR_RATIO = 0.05
GRAD_CLIP = 1.0
GRAD_CKPT = True                      # gradient checkpointing (less memory, ~30% slower)
USE_EMA = True                        # exponential moving average of all trainable weights (dev eval / best ckpt use it)
EMA_DECAY = 0.998
SAVE_EVERY = 300                      # optimizer steps
LOG_EVERY = 50
SEED = 1234

# ---- time budget (free tier!) ----
TIME_BUDGET_H = 9.0                   # wall-clock limit of THIS session (lower it if your quota is smaller)
FINAL_RESERVE_H = 1.0                 # kept free for dev tuning + test prediction (all chunks are scored)
TRAIN_HOURS = 7.5                     # TOTAL training time over all sessions. The cosine LR schedule is TIME-AWARE: it
                                      # anneals to its minimum exactly when this budget (or EPOCHS) is used up, so a
                                      # cut-off run still ends with a properly annealed model.

if SMOKE_TEST:
    N_SMOKE = {"train": 600, "dev": 300, "test": 200}
    EPOCHS, SAVE_EVERY, LOG_EVERY, WARMUP_STEPS = 1, 10, 5, 10
else:
    N_SMOKE = None


# ------------------------------------------------------------------
# small helpers
# ------------------------------------------------------------------
def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def walk_find(filename, roots):
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


HF_SECRET_NAMES = ("HF_TOKEN", "HF", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_API_TOKEN")      # accepted names of the secret


def get_hf_token():
    """Hugging Face token from an environment variable or from a Kaggle secret (Add-ons > Secrets) under any of HF_SECRET_NAMES"""
    for n in HF_SECRET_NAMES:
        if os.environ.get(n, "").strip():
            return os.environ[n].strip()
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
    except Exception:
        return None
    for n in HF_SECRET_NAMES:
        try:
            v = client.get_secret(n)
        except Exception:
            continue
        if v and v.strip():
            return v.strip()
    return None


T_START = time.time()          # session clock: tokenisation / setup time counts against TIME_BUDGET_H

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
else:
    log("WARNING: no GPU visible - this will be extremely slow")


def amp_ctx():
    """mixed precision for the backbone"""
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def no_amp():
    """full fp32 for the (small) hierarchical head: softmax / logsumexp / thresholds stay exact"""
    if AMP["enabled"]:
        return torch.autocast(device_type="cuda", enabled=False)
    return contextlib.nullcontext()


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):                       # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


# ------------------------------------------------------------------
# locate the Phase-1 Model-3 checkpoint (output of the Model-3 pretraining notebook)
# ------------------------------------------------------------------
def find_phase1_ckpt():
    if PHASE1_CKPT:
        return PHASE1_CKPT
    cands, seen = [], []
    for fname, mname in (("best.pt", "best_meta.json"), ("latest.pt", "latest_meta.json")):
        for d in walk_find(fname, [INPUT_ROOT]):
            meta = read_json(os.path.join(d, mname)) or {}
            seen.append((os.path.join(d, fname), meta.get("tag")))
            if meta.get("tag") == PHASE1_MODEL_TAG:
                if fname == "best.pt":
                    cands.append((0, float(meta.get("val_loss", 1e9)), os.path.join(d, fname)))
                else:
                    cands.append((1, -float(meta.get("global_step", 0)), os.path.join(d, fname)))
    if not cands:
        raise FileNotFoundError(
            f"No Model-3 checkpoint (tag '{PHASE1_MODEL_TAG}') under {INPUT_ROOT}. Found: {seen or 'no best.pt/latest.pt at all'}. "
            "Attach the Model-3 pretraining output as an input, or set PHASE1_CKPT (full path of best.pt) manually.")
    cands.sort()                       # best.pt before latest.pt; lowest val_loss first
    return cands[0][2]


assert HEAD_HIDDEN % DOC_HEADS == 0
_ft_name = {"lora": "LoRA", "rslora": "rsLoRA", "lorma": "LoRMA+" if LORMA_VARIANT == "plus" else "LoRMA-pi"}[FT_METHOD]
log("config ready | run tag:", RUN_TAG, "| smoke:", SMOKE_TEST, "| fine-tuning:",
    f"{_ft_name} r={LORA_R}" + ("" if (FT_METHOD == "lorma" and LORMA_VARIANT == "pi") else f" alpha={LORA_ALPHA}"))

In [ ]:
# ============================================================
# CELL 2: MODEL  (Phase-1 Model-3 backbone + LoRMA / LoRA adapters + HIERARCHICAL all-chunk document head)
# The backbone classes below are extracted verbatim from the Model-3 notebook (so state-dict keys, RMSNorm and RoPE
# numerics match the pretrained checkpoint exactly). Model 3 has NO final_norm: its features come from MTP head 1,
#     z = RMSNorm(x + W x),
# which is what we use as the chunk representation (heads 2..K, the t+2 / t+3 predictors, are dropped after loading).
# ============================================================
@dataclass
class ModelConfig:
    vocab_size: int = 50048                       # embedding rows (padded)
    real_vocab: int = 50000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 1536
    dropout: float = 0.1
    max_len: int = 1088
    mtp_heads: int = 3
    mtp_weights: list = field(default_factory=lambda: [1.0, 0.5, 0.25])
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    bos_id: int = 3
    mask_id: int = 4


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.float().pow(2).mean(-1, keepdim=True)
        inv_rms = torch.rsqrt(variance + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def apply_rope(x, cos, sin):
    cos = cos.to(dtype=x.dtype)
    sin = sin.to(dtype=x.dtype)

    x1, x2 = x.chunk(2, dim=-1)

    return torch.cat([
        x1 * cos - x2 * sin,
        x1 * sin + x2 * cos
    ], dim=-1)


def repeat_kv(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd); query head h reads kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

class GQAttention(nn.Module):

    """causal grouped-query self-attention with RoPE (10 query heads share 2 key/value heads)"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class MTPHead(nn.Module):
    """residual projection + RMSNorm:  z = RMSNorm(x + W x).  The logits are z @ E^T with the shared tied embedding E.
    (x is the raw residual stream of the last block; this RMSNorm plays the role of the usual final norm.)"""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm = RMSNorm(dim)

    def forward(self, x):
        return self.norm(x + self.proj(x))


class DecoderLM(nn.Module):
    """inference-time subset of the Model-3 DecoderLM (same parameter names; no init / generation code needed)"""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection of every head
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([MTPHead(cfg.d_model) for _ in range(cfg.mtp_heads)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _rope(self, T):
        return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]

    def hidden(self, idx):
        """idx (B, T) -> residual stream (B, T, D) that all MTP heads read"""
        cos, sin = self._rope(idx.size(1))
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return x

    def head_z(self, x, k):
        return self.heads[k](x)

    def forward(self, idx):
        """next-token logits (B, T, V) from head 1 (used only by the tokenizer sanity check)"""
        return F.linear(self.head_z(self.hidden(idx), 0), self.embed.weight)


BACKBONE = None


def get_backbone():
    """Phase-1 Model-3 weights -> DecoderLM on DEVICE (built once; the CPU copy of the checkpoint is freed)"""
    global BACKBONE
    if BACKBONE is not None:
        return BACKBONE
    assert "model" in PH3, "Phase-1 checkpoint not loaded (run cell 3 first)"
    cfg = ModelConfig(**{k: v for k, v in PH3_CFG.items() if k in ModelConfig.__dataclass_fields__})
    sd = {re.sub(r"^(module\.|_orig_mod\.)+", "", k): v for k, v in PH3["model"].items()}
    lm = DecoderLM(cfg)
    lm.load_state_dict(sd, strict=True)          # fails loudly if architecture / keys differ
    assert CHUNK_LEN <= cfg.max_len, f"CHUNK_LEN {CHUNK_LEN} exceeds the RoPE table of the pretrained model ({cfg.max_len})"
    PH3.pop("model", None)
    PH3.pop("optimizer", None)
    del sd
    gc.collect()
    BACKBONE = lm.to(DEVICE)
    log(f"backbone: {cfg.n_layers} layers, d_model {cfg.d_model}, ffn {cfg.ffn_hidden}, {cfg.n_heads}q/{cfg.n_kv_heads}kv heads, "
        f"{cfg.mtp_heads} MTP heads, vocab rows {cfg.vocab_size} (real {cfg.real_vocab}), max_len {cfg.max_len}")
    return BACKBONE


# ---------------------------- fine-tuning adapters: LoRMA (default) / LoRA / rsLoRA ----------------------------
# FT_METHOD (cell 1) picks the adapter that wraps every target nn.Linear of the frozen backbone; everything else is the same for all of them.
def ft_label():
    return {"lora": "LoRA", "rslora": "rsLoRA", "lorma": "LoRMA+" if LORMA_VARIANT == "plus" else "LoRMA-pi"}[FT_METHOD]


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout, rs=False):
        super().__init__()
        self.base = base
        self.scale = alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))      # B = 0  ->  starts exactly at the pretrained model
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


_ROT_IDX = {}


def rot_index(d, device):
    """idx[i, k] = (k - i) mod d, so gather(M, 1, idx) cyclically shifts row i of M to the right by i (the LoRMA-pi permutation)"""
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device)
        _ROT_IDX[key] = (i[None, :] - i[:, None]) % d
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA: Low-Rank Multiplicative Adaptation. The update MULTIPLIES the frozen weight:  W = M W0  with M = I + s B A  (B: d_out x r,
    A: r x d_out). Computed as  y = W0 x + s B (A (W0 x))  - the d_out x d_out matrix is never formed.
      variant "plus" (LoRMA+, rank inflation by adding the identity):  s = alpha / r, B = 0, A random      -> M = I at step 0
      variant "pi"   (LoRMA-pi, rank inflation by a permutation):      M = P(B A) where row i of B A is cyclically shifted by i; no alpha;
                     B[:, 0] = 1, A[0, 0] = 1, rest of A = 0 and the rest of B random -> P(B A) = I at step 0 (adapter dropout is not used)
    The adapted weight can be merged:  W = (I + s B A) W0  (plus) | P(B A) W0  (pi) - see merged_weight()."""

    def __init__(self, base, r, alpha, dropout, variant="plus"):
        super().__init__()
        assert variant in ("plus", "pi"), f"LORMA_VARIANT must be 'plus' or 'pi', got {variant!r}"
        self.base = base
        self.variant = variant
        d = base.out_features
        self.in_features, self.out_features = base.in_features, d
        r = min(r, d)
        self.r = r
        self.drop = nn.Dropout(dropout) if (dropout > 0 and variant == "plus") else nn.Identity()
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        else:
            self.scale = 1.0
            self.A = nn.Parameter(torch.zeros(r, d))
            self.B = nn.Parameter(torch.empty(d, r))
            nn.init.normal_(self.B, std=1.0 / math.sqrt(r))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def mult_matrix(self):
        """the d_out x d_out matrix M of  W = M W0  (only for merging / tests)"""
        BA = self.B @ self.A
        if self.variant == "plus":
            return torch.eye(self.out_features, dtype=BA.dtype, device=BA.device) + self.scale * BA
        return torch.gather(BA, 1, rot_index(self.out_features, BA.device))

    def merged_weight(self):
        return self.mult_matrix() @ self.base.weight

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)                                    # W0 x
        if self.variant == "plus":
            y = y0 + F.linear(F.linear(self.drop(y0), self.A), self.B) * self.scale
        else:
            P = torch.gather(self.B @ self.A, 1, rot_index(self.out_features, y0.device))
            y = F.linear(y0, P)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


ADAPTER_TYPES = (LoRALinear, LoRMALinear)


def make_adapter(child):
    if FT_METHOD == "lorma":
        return LoRMALinear(child, LORA_R, LORA_ALPHA, LORA_DROPOUT, LORMA_VARIANT)
    if FT_METHOD in ("lora", "rslora"):
        return LoRALinear(child, LORA_R, LORA_ALPHA, LORA_DROPOUT, USE_RSLORA)
    raise ValueError(f"unknown fine-tuning method {FT_METHOD!r} (lorma / rslora / lora)")


def inject_lora(lm):
    """freeze the backbone, set its residual dropout to FT_DROPOUT, wrap every target nn.Linear (attention + SwiGLU of every block) with the adapter"""
    for m in lm.modules():                         # lighter / fine-tuning residual dropout (before the adapter dropouts exist)
        if isinstance(m, nn.Dropout):
            m.p = FT_DROPOUT
    lm.heads = nn.ModuleList([lm.heads[0]])        # keep only the next-token head: its RMSNorm(x + Wx) is our final norm
    for p in lm.parameters():
        p.requires_grad = False
    n = 0
    for blk in lm.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in LORA_TARGETS:
                if hasattr(parent, name) and isinstance(getattr(parent, name), nn.Linear):
                    setattr(parent, name, make_adapter(getattr(parent, name)))
                    n += 1
    lm.heads[0].norm.weight.requires_grad = True   # tiny, helps adapt the pooled representation
    return n


# ---------------------------- hierarchical classifier ----------------------------
PATH_NAMES = ("attention", "chunk-LSE", "mean")


class LSIModel(nn.Module):
    """
    document = ALL its overlapping chunks
      chunk level   : frozen+LoRA decoder -> MTP-head-1 features -> [mean-pool ; last-token] -> MLP
                      -> + chunk-index / relative-position embeddings
      document level: small transformer over the chunk sequence (chunks exchange information, 'section-aware')
      3 aggregation paths, all of which use EVERY chunk:
        (a) label-wise attention : every statute learns which chunks to read           (relevance-weighted pooling)
        (b) chunk-level statute logits, log-mean-exp pooled over chunks                (multiple-instance evidence: one strong
                                                                                        chunk is enough, but all are counted)
        (c) mean of chunk representations -> linear                                    (plain average, every chunk equal)
      a learned per-statute softmax gate fuses the three logits; each path additionally gets its own BCE loss (AUX_W).
    Multi-label: independent sigmoid per statute.
    """

    def __init__(self, lm, n_labels, prior_logits):
        super().__init__()
        self.lm = lm
        d, H = lm.cfg.d_model, HEAD_HIDDEN
        self.chunk_proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(HEAD_DROPOUT),
                                        nn.Linear(2 * d, H), nn.GELU())
        self.pos_emb = nn.Embedding(MAX_POS_EMB, H)           # absolute chunk index
        self.rel_emb = nn.Embedding(REL_BUCKETS + 1, H)       # relative position bucket (+1: single-chunk document)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.normal_(self.rel_emb.weight, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=H, nhead=DOC_HEADS, dim_feedforward=DOC_FF, dropout=HEAD_DROPOUT,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.doc_enc = nn.TransformerEncoder(layer, num_layers=DOC_LAYERS, enable_nested_tensor=False)
        self.doc_norm = nn.LayerNorm(H)
        self.Q = nn.Parameter(torch.randn(n_labels, H) * 0.1)              # (a) label queries over chunks
        self.W_att = nn.Parameter(torch.randn(n_labels, H) * 0.02)
        self.W_chunk = nn.Parameter(torch.randn(n_labels, H) * 0.02)       # (b) chunk-level statute classifier
        self.W_mean = nn.Parameter(torch.randn(n_labels, H) * 0.02)        # (c)
        self.gate = nn.Parameter(torch.zeros(n_labels, 3))                 # softmax -> equal mix at init
        self.b = nn.Parameter(torch.as_tensor(prior_logits, dtype=torch.float32))   # init = log prior odds
        self.grad_ckpt = GRAD_CKPT

    # ---- backbone: one micro-batch of chunks -> (n, 2d) -------------------------------------
    def _encode_mb(self, ids, lens):
        lm = self.lm
        N, T = ids.shape
        cos, sin = lm._rope(T)
        x = lm.drop(lm.embed(ids))
        for blk in lm.blocks:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        x = lm.head_z(x, 0).float()                       # MTP head 1 = RMSNorm(x + W x), plays the role of the final norm
        # right padding + causal attention => real tokens never see pads, so no attention mask is needed
        mask = (torch.arange(T, device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1)
        mean = (x * mask).sum(1) / lens[:, None].float()
        last = x[torch.arange(N, device=ids.device), lens - 1]
        return torch.cat([mean, last], dim=-1)

    def encode_chunks(self, ids, lens):
        """ALL chunks of the batch. Processed in length-sorted micro-batches purely to bound memory."""
        N = ids.size(0)
        mb = ENC_MB_TRAIN if self.training else ENC_MB_EVAL
        if N <= mb:
            return self._encode_mb(ids, lens)
        order = torch.argsort(lens, descending=True)
        outs = []
        for i in range(0, N, mb):
            sel = order[i:i + mb]
            T = int(lens[sel].max().item())
            outs.append(self._encode_mb(ids[sel, :T], lens[sel]))
        out = torch.cat(outs, 0)
        inv = torch.empty_like(order)
        inv[order] = torch.arange(N, device=order.device)
        return out[inv]

    # ---- document head ----------------------------------------------------------------------
    def forward(self, ids, lens, owner, pos, n_docs):
        """returns (logits (D,L), per-path logits incl. bias (D,L,3))"""
        emb = self.encode_chunks(ids, lens)                                   # (N, 2d) fp32
        with no_amp():
            u = self.chunk_proj(emb.float())                                  # (N, H)
            H = u.size(-1)
            counts = torch.bincount(owner, minlength=n_docs)                  # chunks per doc
            Cmax = int(counts.max().item())
            ncd = counts[owner]                                               # chunks of the doc each chunk belongs to
            rb = (((pos.float() + 0.5) / ncd.float()) * REL_BUCKETS).long().clamp(0, REL_BUCKETS - 1)
            rb = torch.where(ncd == 1, torch.full_like(rb, REL_BUCKETS), rb)
            u = u + self.pos_emb(pos.clamp(max=MAX_POS_EMB - 1)) + self.rel_emb(rb)

            U = u.new_zeros(n_docs, Cmax, H).index_put((owner, pos), u)       # (D, C, H) padded
            valid = torch.zeros(n_docs, Cmax, dtype=torch.bool, device=u.device)
            valid[owner, pos] = True
            vm = valid[:, :, None]
            V = self.doc_norm(self.doc_enc(U, src_key_padding_mask=~valid))   # chunks attend to each other
            V = V.masked_fill(~vm, 0.0)                                       # padded slots can never leak (0 * NaN = NaN)

            nv = valid.sum(1).float()                                         # (D,)  real chunks per doc
            # (a) label-wise attention over chunks
            sc = torch.einsum("dch,lh->dcl", V, self.Q) / math.sqrt(H)
            alpha = torch.softmax(sc.masked_fill(~vm, float("-inf")), dim=1)  # (D, C, L)
            z_att = (torch.einsum("dcl,dch->dlh", alpha, V) * self.W_att).sum(-1)
            # (b) chunk-level statute logits, log-mean-exp over the real chunks
            s = torch.einsum("dch,lh->dcl", V, self.W_chunk).masked_fill(~vm, float("-inf"))
            z_pool = torch.logsumexp(s, dim=1) - torch.log(nv)[:, None]
            # (c) mean of chunk representations
            z_mean = ((V * vm).sum(1) / nv[:, None]) @ self.W_mean.t()

            zs = torch.stack([z_att, z_pool, z_mean], dim=-1)                 # (D, L, 3)
            g = torch.softmax(self.gate, dim=-1)                              # (L, 3)
            z = (zs * g[None]).sum(-1) + self.b
            return z, zs + self.b[None, :, None]


def build_model():
    lm = get_backbone()
    n_lora = inject_lora(lm)
    prior = np.clip(Y_train.mean(0), 1e-4, 1 - 1e-4)
    model = LSIModel(lm, N_LABELS, np.log(prior / (1 - prior))).to(DEVICE)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    how = ("identity-initialised multiplicative adapters" if FT_METHOD == "lorma" else "additive adapters")
    scale = (f", scale {LORA_ALPHA / LORA_R:.3f} = alpha/r" if (FT_METHOD == "lorma" and LORMA_VARIANT == "plus")
             else "" if FT_METHOD == "lorma" else f", scale {LORA_ALPHA / (math.sqrt(LORA_R) if USE_RSLORA else LORA_R):.3f}")
    log(f"{ft_label()} modules: {n_lora} ({how}, r={LORA_R}{scale}) | trainable {n_tr / 1e6:.2f}M / total {n_all / 1e6:.1f}M params "
        f"({100 * n_tr / n_all:.1f}%)")
    return model


def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable(model, sd):
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, f"unexpected keys in checkpoint: {unexpected[:5]}"
    train_names = {n for n, p in model.named_parameters() if p.requires_grad}
    assert not (train_names & set(missing)), "some trainable tensors were not restored"

In [ ]:
# ============================================================
# CELL 3: PHASE-1 MODEL-3 CHECKPOINT, DATA (HF), LABELS, DATA AUDIT
# ============================================================
from huggingface_hub import hf_hub_download, login

# ---- Phase-1 Model-3 checkpoint (kept on CPU until the backbone is built) ----
CKPT_PATH = find_phase1_ckpt()
log("Phase-1 checkpoint:", CKPT_PATH)
PH3 = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
PH3_CFG = PH3.get("config") or {}
PH3_VAL = PH3.get("val_loss")                      # next-token CE of head 1 on the pretraining validation windows
_keys = list(PH3["model"].keys())
if any(k.endswith("final_norm.weight") for k in _keys) or not any(".heads." in "." + k or k.startswith("heads.") for k in _keys):
    raise ValueError("this checkpoint has a final_norm and no MTP heads - it looks like a Model-1/2 checkpoint, not Model 3")
log("Phase-1 config:", PH3_CFG)
log(f"Phase-1 pretraining val_loss (head 1): {PH3_VAL} | global_step {PH3.get('global_step')}")
if PH3_CFG:
    log(f"  config says bos_id={PH3_CFG.get('bos_id')} mask_id={PH3_CFG.get('mask_id')} eos_id={PH3_CFG.get('eos_id')} "
        f"-> {'custom word-BPE (bos = mask = eos)' if PH3_CFG.get('bos_id') == PH3_CFG.get('eos_id') else 'HF byte-level BPE (bos=3, mask=4)'} "
        f"was most likely used (verified in the next cell)")

# ---- Phase-5 data from Hugging Face ----
HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN)
    except Exception as e:
        log(f"WARNING: Hugging Face login failed ({type(e).__name__}) - the token is probably wrong or expired")
else:
    log("WARNING: no Hugging Face token found. The Phase-5 dataset is gated: add a Kaggle secret (Add-ons > Secrets) named HF_TOKEN, "
        "tick it for THIS notebook, and make sure that Hugging Face account has been granted access to the dataset.")


def load_records(split):
    try:
        path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE5_FILES[split],
                               repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        if type(e).__name__ in ("GatedRepoError", "RepositoryNotFoundError") or any(c in str(e) for c in ("401", "403")):
            raise RuntimeError(
                f"Cannot download {HF_DATASET_REPO} ({type(e).__name__}). " +
                ("No Hugging Face token was found: add a Kaggle secret named HF_TOKEN (Add-ons > Secrets), tick it for this notebook and re-run. "
                 if not HF_TOKEN else
                 "A token was found but it was rejected: check that it is valid and that this Hugging Face account has been granted access "
                 "to the gated dataset (open the dataset page while logged in and accept / request access). ") +
                "The token itself is read from the secret and never printed.") from e
        raise
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for key in ("data", "records", "examples"):
            if isinstance(obj.get(key), list):
                obj = obj[key]
                break
    assert isinstance(obj, list), f"unexpected JSON structure in {split}"
    log(f"{split:5s}: {len(obj):,} documents")
    return obj


raw = {s: load_records(s) for s in ("train", "dev", "test")}


def parse_labels(lst):
    out = set()
    for x in lst:
        if isinstance(x, bool):
            continue
        if isinstance(x, (int, np.integer)):
            out.add(int(x))
        elif isinstance(x, str) and x.strip().isdigit():
            out.add(int(x.strip()))
    return sorted(out)          # sorted + unique  (3 train docs / 1 dev doc contain duplicate labels)


# label space = every statute id seen in train or dev (100 ids in your EDA)
LABEL_IDS = sorted({l for s in ("train", "dev") for r in raw[s] for l in parse_labels(r["labels"])})
LAB2IDX = {l: i for i, l in enumerate(LABEL_IDS)}
N_LABELS = len(LABEL_IDS)
log(f"{N_LABELS} statutes | id range {LABEL_IDS[0]}..{LABEL_IDS[-1]}")

if SMOKE_TEST:                  # tiny deterministic subsets (labels were computed on the full data above)
    rs = random.Random(SEED)
    for s in raw:
        keep = sorted(rs.sample(range(len(raw[s])), min(N_SMOKE[s], len(raw[s]))))
        raw[s] = [raw[s][i] for i in keep]
        log(f"SMOKE: {s} -> {len(raw[s])} docs")


def build_Y(recs):
    Y = np.zeros((len(recs), N_LABELS), dtype=np.uint8)
    for i, r in enumerate(recs):
        for l in parse_labels(r["labels"]):
            Y[i, LAB2IDX[l]] = 1
    return Y


Y_train, Y_dev = build_Y(raw["train"]), build_Y(raw["dev"])

# ---- label audit: class imbalance / rare statutes / co-occurrence ----
_pos = Y_train.sum(0)
_q = np.percentile(_pos, [0, 25, 50, 75, 100])
log(f"labels/doc: train {Y_train.sum(1).mean():.2f} | dev {Y_dev.sum(1).mean():.2f} | "
    f"single-label docs (train): {(Y_train.sum(1) == 1).mean():.1%}")
log(f"train positives per statute: min {_q[0]:.0f} | p25 {_q[1]:.0f} | median {_q[2]:.0f} | p75 {_q[3]:.0f} | max {_q[4]:.0f}")
log(f"rare statutes (train positives): <10: {int((_pos < 10).sum())} | <25: {int((_pos < 25).sum())} | "
    f"<50: {int((_pos < 50).sum())}  (macro-F1 weights each of them like the frequent ones)")
_o = np.argsort(-_pos)
log("most frequent:", ", ".join(f"{LABEL_IDS[i]}({int(_pos[i])})" for i in _o[:5]),
    "| rarest:", ", ".join(f"{LABEL_IDS[i]}({int(_pos[i])})" for i in _o[-5:]))
_C = Y_train.T.astype(np.int64) @ Y_train.astype(np.int64)
_iu = np.triu_indices(N_LABELS, 1)
_top = np.argsort(-_C[_iu])[:5]
log("top co-occurring statute pairs:", ", ".join(f"({LABEL_IDS[_iu[0][t]]},{LABEL_IDS[_iu[1][t]]}):{int(_C[_iu][t])}" for t in _top))
log(f"statutes with 0 dev positives: {int((Y_dev.sum(0) == 0).sum())}")

In [ ]:
# ============================================================
# CELL 4: TOKENIZER (auto-detected with the pretrained LM's own loss) + TOKENISATION CACHE
# ============================================================
# ---- (A) the tokenizer of the Model-3 notebook, extracted verbatim (word-level BPE, lower-cased, '</w>' markers).
#          NOTE: its WORD_PATTERN contains the character-class range `\a-z`, which silently DROPS ASCII punctuation
#          (. , ; : ( ) ' " - ...). The checkpoint was pretrained on exactly that token stream, so it is reproduced as is.
WORD_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+"""
)


class LegalBPE:
    def __init__(self, vocab_path, merges_path, meta_path=None):
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)

        # Load merge rules
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)

         # Tokenizer fingerprint for window-cache validation
        h = hashlib.md5()

        for path in (vocab_path, merges_path, meta_path):
            if path and os.path.exists(path):
                with open(path, "rb") as f:
                    h.update(f.read())

        self.fingerprint = h.hexdigest()[:12]

        
        # Merge rank: lower rank = earlier BPE merge
        self.ranks = {
            (a, b): i
            for i, (a, b) in enumerate(merges)
        }

        # ID -> token
        self.inv = {
            int(i): t
            for t, i in self.vocab.items()
        }

        self.vocab_size = max(self.inv) + 1

        assert len(self.inv) == self.vocab_size, \
            "vocab ids are not contiguous"

        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        self.bos_id = self.eos_id
        self.mask_id = self.eos_id

        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }
        # Your tokenizer metadata
        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

        for key, val in (
            ("pad_id", self.pad_id),
            ("unk_id", self.unk_id),
            ("eos_id", self.eos_id),
            ("bos_id", self.bos_id),
            ("mask_id", self.mask_id),
        ):
            if key in meta and meta[key] != val:
                raise ValueError(
                    f"meta.json {key}={meta[key]} disagrees with vocab ({val})"
            )

        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(
                f"warning: meta.json vocab_size="
                f"{meta['vocab_size']} but vocab has "
                f"{self.vocab_size}"
            )

        # Decoder-only model
        self.bos_id = self.eos_id
        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }

        self.is_end = [
            self.inv[i].endswith("</w>")
            for i in range(self.vocab_size)
        ]

        # Cache BPE results for repeated words
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit

        # Character sequence + word-end marker
        syms = list(word) + ["</w>"]

        # Apply BPE merges
        while len(syms) > 1:
            best = None
            best_rank = None

            for i in range(len(syms) - 1):
                r = self.ranks.get(
                    (syms[i], syms[i + 1])
                )

                if r is not None and (
                    best_rank is None or r < best_rank
                ):
                    best = (
                        syms[i],
                        syms[i + 1]
                    )
                    best_rank = r

            if best is None:
                break

            merged = best[0] + best[1]

            out = []
            i = 0

            while i < len(syms):
                if (
                    i < len(syms) - 1
                    and syms[i] == best[0]
                    and syms[i + 1] == best[1]
                ):
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1

            syms = out

        ids = [
            self.vocab.get(s, self.unk_id)
            for s in syms
        ]

        # Prevent unlimited cache growth
        if len(self.cache) > 3_000_000:
            self.cache.clear()

        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []

        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))

        return ids

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

    def decode(self, ids):
        toks = [
            self.inv.get(int(i), "")
            for i in ids
            if int(i) not in self.special_ids
        ]

        return "".join(toks).replace("</w>", " ").strip()


class CustomWordBPE(LegalBPE):
    kind = "custom-word-BPE"

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]


# ---- (B) byte-level BPE from the BPE notebook (tokenizer.json, HF `tokenizers`) ----
class HFByteBPE:
    kind = "hf-byte-BPE"

    def __init__(self, path):
        try:
            from tokenizers import Tokenizer
        except ImportError:
            os.system("pip -q install tokenizers")
            from tokenizers import Tokenizer
        self.tokenizer = Tokenizer.from_file(path)
        v = self.tokenizer.get_vocab()
        self.vocab_size = self.tokenizer.get_vocab_size()
        self.pad_id, self.unk_id, self.eos_id = v["<|pad|>"], v["<|unk|>"], v["<|endoftext|>"]
        with open(path, "rb") as f:
            self.fingerprint = hashlib.md5(f.read()).hexdigest()[:12]

    def encode_batch(self, texts):
        return [e.ids for e in self.tokenizer.encode_batch(texts)]


def _file_has_wordend_marker(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            head = f.read(2_000_000)
        return "</w>" in head
    except Exception:
        return False


def discover_tokenizers(max_rows):
    roots = [TOKENIZER_DIR] if TOKENIZER_DIR else [INPUT_ROOT] + ([] if IS_KAGGLE else ["."])
    found, seen = [], set()
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            fs = set(files)
            real = os.path.realpath(d)
            if real in seen:
                continue
            seen.add(real)
            if "vocab_50k.json" in fs and "merges_50k.json" in fs:
                found.append(("custom", d, "vocab_50k.json", "merges_50k.json", "meta_50k.json"))
            elif "vocab.json" in fs and "merges.json" in fs and _file_has_wordend_marker(os.path.join(d, "vocab.json")):
                found.append(("custom", d, "vocab.json", "merges.json", "meta.json"))
            if "tokenizer.json" in fs:
                found.append(("hf", d, "tokenizer.json", None, None))
    cands = []
    for kind, d, a, b, c in found:
        if TOKENIZER_KIND != "auto" and TOKENIZER_KIND != kind:
            continue
        try:
            if kind == "custom":
                t = CustomWordBPE(os.path.join(d, a), os.path.join(d, b), os.path.join(d, c))
            else:
                t = HFByteBPE(os.path.join(d, a))
        except Exception as e:
            log(f"  skip {kind} tokenizer in {d}: {type(e).__name__}: {str(e)[:120]}")
            continue
        if t.vocab_size > max_rows or t.vocab_size < 1000:
            log(f"  skip {t.kind} in {d}: vocab {t.vocab_size} does not fit the model's {max_rows} embedding rows")
            continue
        t.where = d
        cands.append(t)
    return cands


@torch.no_grad()
def lm_next_token_loss(lm, id_lists):
    """mean next-token CE (MTP head 1, tied embedding) of the PRETRAINED backbone on the first CHUNK_LEN tokens of each doc.
    This is exactly the quantity the pretraining val_loss measured, so it is ~val_loss for the tokenizer the model was
    trained with and much larger (~7+) for any other tokenizer."""
    lm.eval()
    tot, cnt = 0.0, 0
    rows = lm.cfg.vocab_size
    for ids in id_lists:
        if len(ids) < CHUNK_LEN + 1:
            continue
        if max(ids[:CHUNK_LEN + 1]) >= rows:
            return float("inf"), 0
        x = torch.tensor(ids[:CHUNK_LEN], dtype=torch.long, device=DEVICE)[None]
        y = torch.tensor(ids[1:CHUNK_LEN + 1], dtype=torch.long, device=DEVICE)
        with amp_ctx():
            logits = lm(x)
        tot += F.cross_entropy(logits[0].float(), y, reduction="sum").item()
        cnt += y.numel()
    return (tot / cnt if cnt else float("nan")), cnt


# ---- text preprocessing (NO character truncation: the whole document is chunked) ----
_PH = re.compile(r"<([A-Z][A-Z_]*)>")
_WS = re.compile(r"\s+")


def prep_text(t):
    if not isinstance(t, str):
        t = ""
    if SIMPLIFY_PLACEHOLDERS:
        t = _PH.sub(r"\1", t)
    return _WS.sub(" ", t).strip()


def select_tokenizer():
    lm = get_backbone()
    rows = lm.cfg.vocab_size
    cands = discover_tokenizers(rows)
    if not cands:
        raise FileNotFoundError(
            "No usable tokenizer under /kaggle/input: need vocab_50k.json + merges_50k.json (Model-3 word-BPE) and/or the "
            "byte-level tokenizer.json from the BPE notebook, with <= %d tokens. Attach the tokenizer output as an input." % rows)
    log("tokenizer candidates:", "; ".join(f"{t.kind} ({t.vocab_size} tokens) in {t.where}" for t in cands))
    if len(cands) == 1 and not TOKENIZER_CHECK:
        return cands[0]
    # sample long training documents for the check (deterministic)
    rs = random.Random(SEED)
    pool = [i for i, r in enumerate(raw["train"]) if isinstance(r.get("text"), str) and len(r["text"]) > 8000]
    pick = rs.sample(pool, min(LM_CHECK_DOCS, len(pool))) if pool else []
    texts = [prep_text(raw["train"][i]["text"]) for i in pick]
    if not texts:
        log("WARNING: no document long enough for the tokenizer check - using the first candidate")
        return cands[0]
    scored = []
    for t in cands:
        loss, n = lm_next_token_loss(lm, t.encode_batch(texts))
        scored.append((loss, t))
        log(f"  {t.kind:16s} -> pretrained-LM next-token loss {loss:.3f} on {n:,} tokens (pretraining val_loss: {PH3_VAL})")
    scored.sort(key=lambda x: (not np.isfinite(x[0]), x[0]))
    best_loss, best = scored[0]
    ref = float(PH3_VAL) if PH3_VAL is not None else 4.5
    if not np.isfinite(best_loss) or best_loss > ref + LM_CHECK_MARGIN:
        msg = (f"best tokenizer ({best.kind}) gives loss {best_loss:.2f} vs pretraining val_loss {ref:.2f}: the tokenizer files "
               f"do not seem to be the ones this checkpoint was trained with (fine-tuning would start from noise). Attach the right "
               f"tokenizer output, or set TOKENIZER_STRICT=False / TOKENIZER_CHECK=False to override.")
        if TOKENIZER_STRICT:
            raise RuntimeError(msg)
        log("WARNING:", msg)
    if len(scored) > 1:
        log(f"  -> using {best.kind} (loss {best_loss:.3f}); the other candidate(s) are ignored")
    return best


tok = select_tokenizer() if TOKENIZER_CHECK else (discover_tokenizers(get_backbone().cfg.vocab_size) or [None])[0]
assert tok is not None, "no tokenizer found"
assert tok.vocab_size <= 65535, "token ids must fit uint16"
PAD_ID = tok.pad_id
log(f"TOKENIZER IN USE: {tok.kind} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id} | "
    f"fingerprint {tok.fingerprint} | {tok.where}")


# ---- tokenisation of the three splits (cached) ----
def cap_tokens(ids):
    """safety cap only (MAX_DOC_TOKENS ~ 52 chunks): keep head + tail of an absurdly long document"""
    if len(ids) <= MAX_DOC_TOKENS:
        return ids, False
    h = int(MAX_DOC_TOKENS * KEEP_HEAD_FRAC)
    return ids[:h] + ids[len(ids) - (MAX_DOC_TOKENS - h):], True


class TokStore:
    """all documents of one split as one flat uint16 array + offsets"""

    def __init__(self, tokens, offsets):
        self.tokens = tokens
        self.offsets = offsets
        self.lens = np.diff(offsets)

    def __len__(self):
        return len(self.lens)


def tokenize_split(name, recs):
    sig = {"v": 3, "n": len(recs), "tok": tok.kind, "tok_fp": tok.fingerprint, "vocab": tok.vocab_size,
           "simplify": SIMPLIFY_PLACEHOLDERS, "max_tokens": MAX_DOC_TOKENS, "head_frac": KEEP_HEAD_FRAC,
           "first": str(recs[0]["id"]), "last": str(recs[-1]["id"])}
    npz, sigp = f"{CACHE_DIR}/{name}_tokens.npz", f"{CACHE_DIR}/{name}_sig.json"
    if os.path.exists(npz) and read_json(sigp) == sig:
        z = np.load(npz)
        log(f"{name}: tokens loaded from cache")
        return TokStore(z["tokens"], z["offsets"])
    t0 = time.time()
    chunks, lens, n_capped = [], [], 0
    BS = 64
    for i in range(0, len(recs), BS):
        texts = [prep_text(r["text"]) for r in recs[i:i + BS]]
        for ids in tok.encode_batch(texts):
            if len(ids) == 0:
                ids = [tok.eos_id]
            ids, capped = cap_tokens(ids)
            n_capped += int(capped)
            chunks.append(np.asarray(ids, dtype=np.uint16))
            lens.append(len(ids))
        if (i // BS) % 40 == 0:
            log(f"  {name}: {i + len(texts):,}/{len(recs):,} docs | {(time.time() - t0) / 60:.1f} min")
    offsets = np.zeros(len(lens) + 1, dtype=np.int64)
    offsets[1:] = np.cumsum(lens)
    tokens = np.concatenate(chunks)
    np.savez(npz, tokens=tokens, offsets=offsets)
    atomic_json(sig, sigp)
    log(f"  {name}: {n_capped} docs hit the {MAX_DOC_TOKENS:,}-token safety cap")
    return TokStore(tokens, offsets)


stores = {s: tokenize_split(s, raw[s]) for s in ("train", "dev", "test")}
for s, st in stores.items():
    q = np.percentile(st.lens, [50, 90, 95, 99])
    log(f"{s:5s}: {len(st):,} docs | tokens/doc median {q[0]:.0f} p90 {q[1]:.0f} p95 {q[2]:.0f} p99 {q[3]:.0f} "
        f"max {st.lens.max():,} | total {st.lens.sum() / 1e6:.1f}M tokens")
_unk = sum(int((st.tokens == tok.unk_id).sum()) for st in stores.values()) / max(sum(len(st.tokens) for st in stores.values()), 1)
log(f"<unk> rate over all splits: {100 * _unk:.3f}%")

TEST_IDS = [r["id"] for r in raw["test"]]          # original ids, original order
for s in raw:                                      # raw texts are no longer needed
    for r in raw[s]:
        r.pop("text", None)
gc.collect()

In [ ]:
# ============================================================
# CELL 5: CHUNK GEOMETRY (all windows), METRICS, THRESHOLD TUNING  (numpy only)
# ============================================================

# ---- overlapping chunk geometry: ALL windows ---------------------------------------
def chunk_starts(n):
    """start offsets of every CHUNK_LEN window (stride STRIDE) needed to cover n tokens; the last window reaches the end"""
    return list(range(0, max(1, n - OVERLAP), STRIDE))


def n_chunks_of(n):
    return -(-max(1, n - OVERLAP) // STRIDE)


def encoded_tokens(n):
    """tokens actually pushed through the backbone for a doc of n tokens (overlap counted twice)"""
    k = n_chunks_of(n)
    return n if k == 1 else (k - 1) * CHUNK_LEN + (n - (k - 1) * STRIDE)


# ---- metrics ----------------------------------------------------------------------
def f1_per_label(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp = (pred & Y).sum(0).astype(np.float64)
    fp = (pred & ~Y).sum(0).astype(np.float64)
    fn = (~pred & Y).sum(0).astype(np.float64)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def all_metrics(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp, fp, fn = (pred & Y).sum(), (pred & ~Y).sum(), (~pred & Y).sum()
    return {"macro_f1": float(f1_per_label(Y, pred).mean()),
            "micro_f1": float(2 * tp / max(2 * tp + fp + fn, 1)),
            "subset_acc": float((pred == Y).all(1).mean()),
            "avg_pred_labels": float(pred.sum(1).mean())}


# ---- thresholds -------------------------------------------------------------------
GRID = np.round(np.arange(0.02, 0.96, 0.01), 2)


def f1_curves(P, Y):
    """(len(GRID), L) matrix: F1 of every label at every global threshold"""
    Yb = Y.astype(bool)
    out = np.zeros((len(GRID), P.shape[1]))
    for gi, t in enumerate(GRID):
        out[gi] = f1_per_label(Yb, P >= t)
    return out


def _smooth(curves, w=5):
    pad = w // 2
    c = np.pad(curves, ((pad, pad), (0, 0)), mode="edge")
    k = np.ones(w) / w
    return np.stack([np.convolve(c[:, j], k, mode="valid") for j in range(c.shape[1])], 1)


def fit_thresholds(P, Y, mode="blend", lam=0.5):
    """mode: global (one t) | perlabel (one t per label, smoothed F1 curve) | blend (lam*perlabel + (1-lam)*global)"""
    curves = f1_curves(P, Y)
    g = GRID[np.argmax(curves.mean(1))]
    t_l = GRID[np.argmax(_smooth(curves), 0)]
    if mode == "global":
        return np.full(P.shape[1], g)
    if mode == "perlabel":
        return t_l.astype(np.float64)
    return lam * t_l + (1 - lam) * g


def apply_thresholds(P, thr, force_top1=True):
    """multi-hot prediction; every document has >=1 statute in train/dev, so empty rows get their best-ratio label"""
    pred = P >= thr[None, :]
    if force_top1:
        empty = np.where(~pred.any(1))[0]
        if len(empty):
            pred[empty, np.argmax(P[empty] / thr[None, :], 1)] = True
    return pred


def crossfit_report(P, Y, modes=("global", "perlabel", "blend"), folds=2, seed=0):
    """honest dev estimate: thresholds fitted on the other half of dev, scored on this half"""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(P))
    parts = np.array_split(perm, folds)
    res = {}
    for mode in modes:
        pred = np.zeros(P.shape, dtype=bool)
        for k in range(folds):
            te = parts[k]
            tr = np.concatenate([parts[j] for j in range(folds) if j != k])
            thr = fit_thresholds(P[tr], Y[tr], mode)
            pred[te] = apply_thresholds(P[te], thr)
        res[mode] = all_metrics(Y, pred)
    return res


# ---- coverage audit: every token of every document is inside at least one chunk ----
log("chunk audit (EVERY chunk of EVERY document is encoded - no sampling anywhere):")
for _s, _st in stores.items():
    _n = np.array([n_chunks_of(int(x)) for x in _st.lens])
    assert all((k - 1) * STRIDE + CHUNK_LEN >= x for k, x in zip(_n, _st.lens)), "chunk geometry leaves a gap"
    assert _n.max() <= MAX_POS_EMB, f"{_n.max()} chunks > MAX_POS_EMB={MAX_POS_EMB}"
    _enc = sum(encoded_tokens(int(x)) for x in _st.lens)
    log(f"  {_s:5s}: chunks/doc median {np.median(_n):.0f} p90 {np.percentile(_n, 90):.0f} p99 {np.percentile(_n, 99):.0f} "
        f"max {_n.max()} | total chunks {_n.sum():,} | {_enc / 1e6:.1f}M tokens through the backbone per pass "
        f"({_enc / max(_st.lens.sum(), 1):.2f}x overlap factor)")

In [ ]:
# ============================================================
# CELL 6: BATCHING (token-budget, length-bucketed, ALL chunks) AND INFERENCE
# ============================================================
def doc_cost(n_tokens):
    """padded tokens a doc costs in a batch: (#chunks) x (chunk length). No cap: every chunk is used."""
    return n_chunks_of(n_tokens) * min(CHUNK_LEN, n_tokens)


def pack(doc_order, store, budget, max_docs):
    """greedy packing; a document is NEVER split or cut - one that exceeds the budget simply gets a batch of its own"""
    batches, cur, cost = [], [], 0
    for d in doc_order:
        c = doc_cost(int(store.lens[d]))
        if cur and (cost + c > budget or len(cur) >= max_docs):
            batches.append(cur)
            cur, cost = [], 0
        cur.append(int(d))
        cost += c
    if cur:
        batches.append(cur)
    return batches


def make_train_batches(store, epoch):
    """deterministic given (SEED, epoch): shuffle -> sort by length inside pools (less padding) -> pack -> shuffle batches"""
    rng = np.random.RandomState(SEED + 1000 * epoch)
    order = rng.permutation(len(store))
    POOL = 1536
    batches = []
    for p in range(0, len(order), POOL):
        pool = sorted(order[p:p + POOL], key=lambda d: store.lens[d])
        batches += pack(pool, store, TRAIN_TOKEN_BUDGET, MAX_DOCS_PER_BATCH)
    perm = rng.permutation(len(batches))
    return [batches[i] for i in perm]


def make_eval_batches(store):
    order = np.argsort(store.lens)
    return pack(order, store, EVAL_TOKEN_BUDGET, 128)


def collate(store, doc_ids):
    """every chunk of every document in doc_ids -> padded ids + (owner doc, chunk position) bookkeeping"""
    seqs, owner, pos = [], [], []
    for bi, d in enumerate(doc_ids):
        a, b = int(store.offsets[d]), int(store.offsets[d + 1])
        n = b - a
        for p, s in enumerate(chunk_starts(n)):
            seqs.append(store.tokens[a + s: a + min(n, s + CHUNK_LEN)])
            owner.append(bi)
            pos.append(p)
    lens = np.array([len(x) for x in seqs], dtype=np.int64)
    ids = np.full((len(seqs), int(lens.max())), PAD_ID, dtype=np.int64)
    for i, x in enumerate(seqs):
        ids[i, :len(x)] = x
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return (t(ids), t(lens), t(np.asarray(owner, dtype=np.int64)), t(np.asarray(pos, dtype=np.int64)), len(doc_ids))


@torch.no_grad()
def predict_probs(model, store, name="", paths=False):
    """sigmoid probabilities (n_docs, N_LABELS) from ALL chunks of every document, rows aligned with the store.
    paths=True additionally returns {path name: probabilities} for the 3 aggregation paths (ablation)."""
    model.eval()
    P = np.zeros((len(store), N_LABELS), dtype=np.float32)
    PP = np.zeros((len(store), N_LABELS, 3), dtype=np.float32) if paths else None
    batches = make_eval_batches(store)
    t0, done = time.time(), 0
    for i, docs in enumerate(batches):
        ids, lens, owner, pos, nd = collate(store, docs)
        with amp_ctx():
            logits, zs = model(ids, lens, owner, pos, nd)
        P[docs] = torch.sigmoid(logits.float()).cpu().numpy()
        if paths:
            PP[docs] = torch.sigmoid(zs.float()).cpu().numpy()
        done += len(docs)
        if i % 50 == 0:
            log(f"  predict {name}: {done:,}/{len(store):,} docs | {(time.time() - t0) / 60:.1f} min")
    if paths:
        return P, {PATH_NAMES[k]: PP[:, :, k] for k in range(3)}
    return P

In [ ]:
# ============================================================
# CELL 7: TRAINING  (all chunks, resumable, time-aware LR schedule, EMA, best epoch chosen on dev macro-F1)
# Checkpoint files: lsi_latest.pt / lsi_latest_meta.json (resume) and lsi_best.pt / lsi_best_meta.json
# ============================================================
# pos_weight: mild up-weighting of positives for rare statutes (sqrt of the neg/pos ratio, capped)
_pos_cnt = Y_train.sum(0).astype(np.float64)
POS_W = torch.tensor(np.clip(np.sqrt((len(Y_train) - _pos_cnt) / np.maximum(_pos_cnt, 1)), 1.0, POS_WEIGHT_CAP),
                     dtype=torch.float32, device=DEVICE)
log(f"pos_weight range: {POS_W.min().item():.2f} .. {POS_W.max().item():.2f}")

Y_train_t = torch.from_numpy(Y_train.astype(np.float32))


def make_optimizer(model):
    groups = {"lora": [], "norm": [], "head_w": [], "head_nd": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("lm.") and "heads.0.norm" in n:      # Model 3: MTP-head-1 RMSNorm = final norm
            groups["norm"].append(p)
        elif n.startswith("lm."):
            groups["lora"].append(p)
        elif p.ndim >= 2:
            groups["head_w"].append(p)
        else:
            groups["head_nd"].append(p)
    # LoRMA-pi: A and B are not weight-decayed (decaying A would shrink M = P(BA) towards 0, not towards the identity)
    wd_ad = 0.0 if (FT_METHOD == "lorma" and LORMA_VARIANT == "pi") else WEIGHT_DECAY
    spec = [("lora", LORA_LR, wd_ad), ("norm", LORA_LR, 0.0),
            ("head_w", HEAD_LR, WEIGHT_DECAY), ("head_nd", HEAD_LR, 0.0)]
    pg = [{"params": groups[k], "lr": lr, "base_lr": lr, "weight_decay": wd} for k, lr, wd in spec if groups[k]]
    return torch.optim.AdamW(pg, betas=(0.9, 0.99), eps=1e-8, fused=(DEVICE.type == "cuda"))


class EMA:
    """exponential moving average of the trainable weights (adapters A / B + head). `applied()` temporarily swaps it in."""

    def __init__(self, model, decay):
        self.names = [n for n, p in model.named_parameters() if p.requires_grad]
        self.params = [p for n, p in model.named_parameters() if p.requires_grad]
        self.shadow = [p.detach().clone().float() for p in self.params]
        self.decay = decay

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        torch._foreach_mul_(self.shadow, d)
        torch._foreach_add_(self.shadow, [p.detach().float() for p in self.params], alpha=1.0 - d)

    @contextlib.contextmanager
    def applied(self):
        backup = [p.detach().clone() for p in self.params]
        with torch.no_grad():
            for p, s in zip(self.params, self.shadow):
                p.copy_(s)
        try:
            yield
        finally:
            with torch.no_grad():
                for p, b in zip(self.params, backup):
                    p.copy_(b)

    def state_dict(self):
        return {n: s.detach().cpu().clone() for n, s in zip(self.names, self.shadow)}

    def load_state_dict(self, sd):
        for n, s in zip(self.names, self.shadow):
            s.copy_(sd[n].to(s.device))


def lr_mult(step, prog):
    """linear warm-up (by steps) x cosine decay driven by `prog` = max(step progress, wall-clock progress)"""
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))


FT_SETTINGS = {"method": FT_METHOD, "variant": LORMA_VARIANT if FT_METHOD == "lorma" else None, "r": LORA_R,
               "alpha": None if (FT_METHOD == "lorma" and LORMA_VARIANT == "pi") else LORA_ALPHA,
               "targets": list(LORA_TARGETS), "lora_lr": LORA_LR, "head_lr": HEAD_LR}


def check_ft_compat(meta):
    """a checkpoint only makes sense for the fine-tuning setting that wrote it (adapter tensors of LoRMA+ and LoRMA-pi, or of another alpha,
    load without any error but mean something else)"""
    old = (meta or {}).get("ft")
    if old is None:
        return
    for k in ("method", "variant", "r", "alpha", "targets"):
        if old.get(k) != FT_SETTINGS.get(k):
            raise RuntimeError(f"a checkpoint with this RUN_TAG was trained with fine-tuning setting {k} = {old.get(k)!r}, this notebook has "
                               f"{FT_SETTINGS.get(k)!r}. Use the matching settings, or a new RUN_TAG for a fresh run.")


def find_previous_latest():
    best = None
    for d in walk_find("lsi_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_latest_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or meta.get("n_labels") != N_LABELS:
            continue
        if not os.path.exists(os.path.join(d, "lsi_latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def carry_over_best():
    """copy the best checkpoint of an earlier session (attached as input) into this session's checkpoint dir"""
    cur = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
    for d in walk_find("lsi_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_best_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or not os.path.exists(os.path.join(d, "lsi_best.pt")):
            continue
        check_ft_compat(meta)
        if cur is None or meta["dev_macro_f1"] > cur["dev_macro_f1"]:
            shutil.copy(os.path.join(d, "lsi_best.pt"), f"{CKPT_DIR}/lsi_best.pt.tmp")
            os.replace(f"{CKPT_DIR}/lsi_best.pt.tmp", f"{CKPT_DIR}/lsi_best.pt")
            shutil.copy(os.path.join(d, "lsi_best_meta.json"), f"{CKPT_DIR}/lsi_best_meta.json")
            cur = meta
    return cur


def dev_eval(model):
    """dev macro-F1 with the best single global threshold (cheap, comparable across epochs); all chunks of all dev docs"""
    P = predict_probs(model, stores["dev"], "dev")
    thr = fit_thresholds(P, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(P, thr))
    log(f"  DEV  macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f} "
        f"| global thr {thr[0]:.2f} | avg labels/doc {m['avg_pred_labels']:.2f}")
    return m


def train(model):
    best_meta = carry_over_best()
    best_f1 = best_meta["dev_macro_f1"] if best_meta else -1.0
    opt = make_optimizer(model)
    scaler = make_grad_scaler(AMP["enabled"] and AMP["dtype"] == torch.float16)
    state = {"epoch": 0, "bi": 0, "global_step": 0, "train_sec": 0.0}
    ck_ema = None

    prev = find_previous_latest()
    if prev is not None:
        d, meta = prev
        check_ft_compat(meta)
        ck = torch.load(os.path.join(d, "lsi_latest.pt"), map_location="cpu", weights_only=False)
        load_trainable(model, ck["trainable"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        state.update(epoch=ck["epoch"], bi=ck["bi"], global_step=ck["global_step"], train_sec=ck.get("train_sec", 0.0))
        ck_ema = ck.get("ema")
        log(f"RESUMED from {d}: epoch {state['epoch'] + 1}, batch {state['bi']}, step {state['global_step']}, "
            f"train time so far {fmt_time(state['train_sec'])}")
        del ck
    ema = EMA(model, EMA_DECAY) if USE_EMA else None
    if ema is not None and ck_ema is not None:
        ema.load_state_dict(ck_ema)
    train_params = [p for g in opt.param_groups for p in g["params"]]

    n_batches0 = len(make_train_batches(stores["train"], 0))
    total_steps = EPOCHS * n_batches0
    n_chunks_train = sum(n_chunks_of(int(n)) for n in stores["train"].lens)
    log(f"{len(stores['train']):,} train docs | {n_chunks_train:,} chunks per epoch (ALL used) | ~{n_batches0:,} steps/epoch "
        f"| {EPOCHS} epochs | ~{total_steps:,} steps")

    # time-aware schedule: the LR reaches its minimum when EPOCHS finish OR the training-time budget is used up
    sess_avail = (TIME_BUDGET_H - FINAL_RESERVE_H) * 3600 - (time.time() - T_START)
    train_budget = min(TRAIN_HOURS * 3600, state["train_sec"] + max(sess_avail, 0.0))
    log(f"training-time budget: {fmt_time(train_budget)} in total ({fmt_time(state['train_sec'])} already used) | "
        f"session wall-clock limit {TIME_BUDGET_H}h, {FINAL_RESERVE_H}h reserved for final prediction")

    last = [time.time()]

    def tick():
        now = time.time()
        state["train_sec"] += now - last[0]
        last[0] = now

    def progress():
        return min(1.0, max(state["global_step"] / total_steps, state["train_sec"] / max(train_budget, 1.0)))

    def save_latest():
        atomic_torch_save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "ema": ema.state_dict() if ema is not None else None,
                           "epoch": state["epoch"], "bi": state["bi"], "global_step": state["global_step"],
                           "train_sec": state["train_sec"], "tag": RUN_TAG}, f"{CKPT_DIR}/lsi_latest.pt")
        atomic_json({"tag": RUN_TAG, "n_labels": N_LABELS, "epoch": state["epoch"], "bi": state["bi"],
                     "global_step": state["global_step"], "train_sec": state["train_sec"],
                     "ft": FT_SETTINGS, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f"{CKPT_DIR}/lsi_latest_meta.json")

    def eval_and_maybe_save_best(tag):
        nonlocal best_f1
        with (ema.applied() if ema is not None else contextlib.nullcontext()):      # dev score of the EMA weights
            m = dev_eval(model)
            if m["macro_f1"] > best_f1:
                best_f1 = m["macro_f1"]
                atomic_torch_save({"trainable": trainable_state(model), "dev_macro_f1": best_f1, "tag": RUN_TAG},
                                  f"{CKPT_DIR}/lsi_best.pt")
                atomic_json({"tag": RUN_TAG, "dev_macro_f1": best_f1, "where": tag, "global_step": state["global_step"],
                             "label_ids": LABEL_IDS, "n_labels": N_LABELS, "ft": FT_SETTINGS}, f"{CKPT_DIR}/lsi_best_meta.json")
                log(f"  >> NEW BEST dev macro-F1 {best_f1:.4f} ({tag}) - lsi_best.pt saved")
        return m

    timed_out = False
    t_loop = time.time()
    steps_this_run, chunks_seen, tokens_seen, run_loss, run_n, n_skipped = 0, 0, 0, 0.0, 0, 0

    for epoch in range(state["epoch"], EPOCHS):
        batches = make_train_batches(stores["train"], epoch)
        model.train()
        for bi in range(state["bi"], len(batches)):
            if state["train_sec"] >= train_budget:
                timed_out = True
                break
            docs = batches[bi]
            ids, lens, owner, pos, nd = collate(stores["train"], docs)       # ALL chunks of every doc in the batch
            y = Y_train_t[docs].to(DEVICE)

            mult = lr_mult(state["global_step"], progress())
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * mult

            with amp_ctx():
                logits, zs = model(ids, lens, owner, pos, nd)

            def bce(z):
                return F.binary_cross_entropy_with_logits(z.float(), y, pos_weight=POS_W, reduction="none").mean(1)

            loss_main = bce(logits)
            loss_aux = sum(bce(zs[:, :, k]) for k in range(3)) / 3.0         # every path must be useful on its own
            loss = (loss_main + AUX_W * loss_aux).sum() / DOC_NORM
            opt.zero_grad(set_to_none=True)
            if torch.isfinite(loss):
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(train_params, GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                if ema is not None:
                    ema.update(state["global_step"])
                run_loss += loss_main.mean().item()
                run_n += 1
            else:
                n_skipped += 1
                log(f"  WARNING: non-finite loss at step {state['global_step']} (batch of {len(docs)} docs) - step skipped")

            state["global_step"] += 1
            state["epoch"], state["bi"] = epoch, bi + 1
            steps_this_run += 1
            chunks_seen += int(lens.numel())
            tokens_seen += int(lens.sum().item())
            tick()

            if steps_this_run % LOG_EVERY == 0:
                el = time.time() - t_loop
                eta_epoch = (len(batches) - bi - 1) * el / steps_this_run
                log(f"ep {epoch + 1}/{EPOCHS} step {bi + 1}/{len(batches)} (global {state['global_step']}) | "
                    f"loss {run_loss / max(run_n, 1):.4f} | lr x{mult:.3f} | {chunks_seen / el:.1f} chunks/s "
                    f"{tokens_seen / el / 1000:.1f}k tok/s | epoch ETA {fmt_time(eta_epoch)} | "
                    f"schedule {100 * progress():.0f}% | train time {fmt_time(state['train_sec'])}/{fmt_time(train_budget)}")
                run_loss, run_n = 0.0, 0
            if state["global_step"] % SAVE_EVERY == 0:
                save_latest()
                tick()

        if timed_out:
            log(f"training-time budget reached inside epoch {epoch + 1} - saving, then dev tuning / prediction")
            save_latest()
            eval_and_maybe_save_best(f"timeout@epoch{epoch + 1}")
            tick()
            break
        state["epoch"], state["bi"] = epoch + 1, 0
        save_latest()
        eval_and_maybe_save_best(f"epoch{epoch + 1}")
        tick()
        model.train()

    if n_skipped:
        log(f"note: {n_skipped} step(s) skipped because of a non-finite loss")
    log(f"training finished | best dev macro-F1 (global thr) {best_f1:.4f} | train time {fmt_time(state['train_sec'])} "
        f"| session elapsed {fmt_time(time.time() - T_START)}")
    return best_f1


model = build_model()
BEST_DEV_F1 = train(model)

In [ ]:
# ============================================================
# CELL 8: BEST CHECKPOINT -> PATH ABLATION -> DEV THRESHOLD TUNING -> TEST PREDICTION -> submission.zip
# (every chunk of every dev / test document is scored)
# ============================================================
bm = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
assert bm is not None and os.path.exists(f"{CKPT_DIR}/lsi_best.pt"), "no best checkpoint found - did training run?"
assert bm["label_ids"] == LABEL_IDS, "label space differs from the one the checkpoint was trained with"
ck = torch.load(f"{CKPT_DIR}/lsi_best.pt", map_location="cpu", weights_only=False)
load_trainable(model, ck["trainable"])
log(f"loaded best checkpoint (dev macro-F1 {ck['dev_macro_f1']:.4f}, {bm['where']})")

# ---- 1) dev probabilities (fused + the 3 aggregation paths) ----
P_dev, P_dev_paths = predict_probs(model, stores["dev"], "dev", paths=True)

log("\nAggregation-path ablation on dev (global threshold, >=1 label/doc; paths were trained jointly, so this shows what each adds):")
for name, Pp in list(P_dev_paths.items()) + [("FUSED (used)", P_dev)]:
    thr_g = fit_thresholds(Pp, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(Pp, thr_g))
    log(f"  {name:13s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f}")

cf = crossfit_report(P_dev, Y_dev)          # thresholds fitted on one half of dev, scored on the other half
log("\nDEV, honest cross-fitted estimate of the fused model (force >=1 label per document):")
for mode, m in cf.items():
    log(f"  {mode:9s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | "
        f"subset-acc {m['subset_acc']:.4f} | avg labels/doc {m['avg_pred_labels']:.2f}")
BEST_MODE = max(cf, key=lambda k: cf[k]["macro_f1"])
THR = fit_thresholds(P_dev, Y_dev, BEST_MODE)          # final thresholds use all of dev
pred_dev = apply_thresholds(P_dev, THR)
m_dev = all_metrics(Y_dev, pred_dev)
log(f"chosen strategy: {BEST_MODE} | in-sample dev macro-F1 {m_dev['macro_f1']:.4f} (optimistic, as thresholds were fitted on it)")
log(f"thresholds: min {THR.min():.2f} | median {np.median(THR):.2f} | max {THR.max():.2f}")

f1_l = f1_per_label(Y_dev, pred_dev)
worst = np.argsort(f1_l)[:8]
log("weakest statutes on dev (id: F1, train positives): " +
    ", ".join(f"{LABEL_IDS[i]}: {f1_l[i]:.2f} ({int(Y_train[:, i].sum())})" for i in worst))

# ---- 2) test prediction ----
P_test = predict_probs(model, stores["test"], "test")
pred_test = apply_thresholds(P_test, THR)

preds = []
valid_ids = set(LABEL_IDS)
for i, doc_id in enumerate(TEST_IDS):
    ids = sorted({int(LABEL_IDS[j]) for j in np.where(pred_test[i])[0]})       # unique ints, valid statute ids only
    assert ids and all(x in valid_ids for x in ids)
    preds.append({"id": doc_id, "pred": ids})

assert len(preds) == len(TEST_IDS) == len(stores["test"])
assert len({str(p["id"]) for p in preds}) == len(preds), "duplicate test ids"
log(f"test: {len(preds):,} predictions | avg statutes/doc {np.mean([len(p['pred']) for p in preds]):.2f} "
    f"(dev truth: {Y_dev.sum(1).mean():.2f})")

# ---- 3) save everything ----
pred_path = f"{OUT_DIR}/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = f"{OUT_DIR}/submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")            # prediction.json at the zip root

np.save(f"{OUT_DIR}/dev_probs.npy", P_dev)                   # keep for ensembling with the other Phase-1 models
np.save(f"{OUT_DIR}/test_probs.npy", P_test)
atomic_json({"label_ids": LABEL_IDS, "thresholds": [float(x) for x in THR], "strategy": BEST_MODE,
             "dev_crossfit": cf, "dev_in_sample": m_dev, "test_ids": [str(x) for x in TEST_IDS],
             "tokenizer": {"kind": tok.kind, "fingerprint": tok.fingerprint, "where": tok.where}, "run_tag": RUN_TAG,
             "fine_tuning": {**FT_SETTINGS, "label": ft_label()}},
            f"{OUT_DIR}/thresholds_and_report.json")
log("saved:", pred_path, "|", zip_path, "| dev_probs.npy | test_probs.npy | thresholds_and_report.json")
log("done.")